# 🏺 Egyptian Hieroglyph Recognition — Full Pipeline v5.0
### Fuentes-Ferrer et al. (2025) · Graduation Project · Google Colab Edition

| Step | Module | Key Techniques |
|------|--------|---------------|
| 1 | **Data Pre-processing** | Glyph2025 merge · letterbox resize · 164-class filter |
| 2 | **SAM Fine-tuning** | Per-instance masks · LR warmup · batch image encoder |
| 3 | **Segmentation (IGSM)** | Multi-threshold · stroke detect · vectorised NMS · RTL order |
| 4 | **Classification (CVV)** | ConvNeXt-Tiny · 5-fold · MixUp · 7-view TTA · weighted ensemble |
| 5 | **Inference** | End-to-end · `inference_mode` · graceful error handling |

### v5.0 — All Fixes Applied
| # | Fix | Detail |
|---|-----|--------|
| 1 | **Vectorised NMS** | bbox IoU pre-filter → O(n) instead of O(n²) mask comparison |
| 2 | **Dataset read-once** | Image loaded once per `__getitem__`, cached handle in worker |
| 3 | **pin_memory guard** | Auto-disabled when RAM < 8 GB to prevent silent OOM |
| 4 | **Model load robust** | Architecture mismatch caught with clear error on `load_state_dict` |
| 5 | **inference_mode** | Replaces `no_grad` everywhere in inference — faster + less memory |
| 6 | **Colab-ready** | Paths, `kaggle.json` setup, Drive mount, GPU check all automated |

**Dataset:** https://www.kaggle.com/datasets/ahmedelkelany/egyptian-hieroglyphic-signs-segmentation  
**Paper:** https://doi.org/10.1016/j.asoc.2025.112793  
---

## ▶ Cell 0 — Colab Setup & Dependencies
> Run **once**. Handles Kaggle API, dataset download, and all pip installs.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 0 — Colab Setup
#  Dataset 1 (Segmentation) : Kaggle → ahmedelkelany/egyptian-hieroglyphic-signs-segmentation
#  Dataset 2 (Classification): GitHub → rfuentesfe/EgyptianHieroglyphicText
# ══════════════════════════════════════════════════════════════════════
import subprocess, sys, os, json, shutil
import torch

# ── GPU check ─────────────────────────────────────────────────────────
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(f"🖥️  Runtime : {gpu}")
if not torch.cuda.is_available():
    print("⚠️  No GPU — Runtime → Change runtime type → T4 GPU")

# ── Packages ──────────────────────────────────────────────────────────
def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

try:
    import segment_anything
    print("✅ segment_anything already installed")
except ImportError:
    _pip("git+https://github.com/facebookresearch/segment-anything.git")

for pkg in ["kaggle", "opencv-python-headless", "scikit-image",
            "albumentations", "tqdm", "psutil"]:
    _pip(pkg)

# ── Kaggle credentials ────────────────────────────────────────────────
KAGGLE_USERNAME = ""   # ← حط username بتاعك (من Kaggle → Account → API)
KAGGLE_KEY      = ""   # ← حط key بتاعك

KAGGLE_JSON = "/root/.kaggle/kaggle.json"
os.makedirs("/root/.kaggle", exist_ok=True)

if not os.path.exists(KAGGLE_JSON):
    if KAGGLE_USERNAME and KAGGLE_KEY:
        with open(KAGGLE_JSON, "w") as f:
            json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
        os.chmod(KAGGLE_JSON, 0o600)
        print("✅ kaggle.json written")
    else:
        try:
            from google.colab import files
            print("📂 Upload your kaggle.json ...")
            uploaded = files.upload()
            if "kaggle.json" in uploaded:
                with open(KAGGLE_JSON, "wb") as f: f.write(uploaded["kaggle.json"])
                os.chmod(KAGGLE_JSON, 0o600)
                print("✅ kaggle.json uploaded")
        except Exception as e:
            print(f"⚠️  {e} — set KAGGLE_USERNAME/KEY above")
else:
    print("✅ kaggle.json found")

# ── Paths ─────────────────────────────────────────────────────────────
WORK_DIR        = "/content/hiero"
SEG_DATA_DIR    = os.path.join(WORK_DIR, "seg_dataset")    # Segmentation dataset
CLASS_DATA_DIR  = os.path.join(WORK_DIR, "class_dataset")  # Classification dataset
os.makedirs(SEG_DATA_DIR,   exist_ok=True)
os.makedirs(CLASS_DATA_DIR, exist_ok=True)

# ── Dataset 1: Segmentation (Kaggle) ─────────────────────────────────
SEG_DATASET_ID = "ahmedelkelany/egyptian-hieroglyphic-signs-segmentation"
if not os.listdir(SEG_DATA_DIR):
    if os.path.exists(KAGGLE_JSON):
        print("⏳ Downloading segmentation dataset ...")
        subprocess.run(
            f"kaggle datasets download -d {SEG_DATASET_ID} -p {SEG_DATA_DIR} --unzip",
            shell=True, check=True
        )
        print(f"✅ Segmentation dataset → {SEG_DATA_DIR}")
    else:
        print("❌ kaggle.json not found")
else:
    n = sum(len(fs) for _,_,fs in os.walk(SEG_DATA_DIR))
    print(f"✅ Segmentation dataset exists ({n:,} files)")

# ── Dataset 2: Classification (GitHub) ───────────────────────────────
# rfuentesfe/EgyptianHieroglyphicText — فيه class folders بأسماء Gardiner codes
HIERO_REPO = "https://github.com/rfuentesfe/EgyptianHieroglyphicText.git"
HIERO_DIR  = os.path.join(CLASS_DATA_DIR, "EgyptianHieroglyphicText")

if not os.path.exists(HIERO_DIR):
    print("⏳ Cloning classification dataset from GitHub ...")
    subprocess.run(
        ["git", "clone", "--depth=1", HIERO_REPO, HIERO_DIR],
        check=True, capture_output=True
    )
    print(f"✅ Classification dataset → {HIERO_DIR}")
    # Show top-level structure
    print(f"   Contents: {os.listdir(HIERO_DIR)[:8]}")
else:
    print(f"✅ Classification dataset exists → {HIERO_DIR}")

# ── Drive (optional) ──────────────────────────────────────────────────
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/hiero_checkpoints"
    os.makedirs(WORK_DIR, exist_ok=True)
    print(f"✅ Drive mounted → {WORK_DIR}")

print("\n✅ Cell 0 complete!")


---
# 📦 STEP 1 — Data Pre-Processing
> Load dataset · build Glyph2025 · letterbox resize · 164-class filter

### Cell 1.1 — Imports & Global Config

In [ ]:
import os, random, json, time, warnings, shutil, subprocess
import tarfile, urllib.request, gc, re, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
import functools
from PIL import Image
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support)
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
warnings.filterwarnings("ignore")

# ── Logging ────────────────────────────────────────────────────────────
for _h in logging.getLogger("hiero").handlers[:]:
    logging.getLogger("hiero").removeHandler(_h)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("hiero")

# ── Reproducibility ────────────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

# ── Paths ──────────────────────────────────────────────────────────────
WORK_DIR        = "/content/hiero"
SEG_DATA_DIR    = os.path.join(WORK_DIR, "seg_dataset")    # Segmentation (Kaggle)
CLASS_DATA_DIR  = os.path.join(WORK_DIR, "class_dataset")  # Classification (GitHub)
GLYPH2025_RAW   = os.path.join(WORK_DIR, "Glyph2025_raw")
GLYPH2025_DIR   = os.path.join(WORK_DIR, "Glyph2025_processed")
SEG_OUTPUT_DIR  = os.path.join(WORK_DIR, "segmented_glyphs")
SAM_CKPT        = os.path.join(WORK_DIR, "sam_vit_b.pth")
SAM_FINETUNE_DIR= os.path.join(WORK_DIR, "sam_finetuned")
SAM_BEST_PATH   = os.path.join(SAM_FINETUNE_DIR, "best_sam_hiero.pth")
SAM_LAST_PATH   = os.path.join(SAM_FINETUNE_DIR, "last_sam_hiero.pth")
STEP1_STATE     = os.path.join(WORK_DIR, "step1_state.json")
STEP2_STATE     = os.path.join(WORK_DIR, "step2_state.json")
TRAINING_STATE  = os.path.join(WORK_DIR, "training_state.json")

# DATA_DIR للـ SAM fine-tuning (segmentation dataset)
DATA_DIR   = SEG_DATA_DIR
STELAE_DIR = SEG_DATA_DIR

for d in [SEG_OUTPUT_DIR, SAM_FINETUNE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Image settings ─────────────────────────────────────────────────────
IMG_SIZE   = 224
IMG_EXTS   = (".jpg",".jpeg",".png",".bmp",".webp")
MEAN       = [0.485, 0.456, 0.406]
STD        = [0.229, 0.224, 0.225]
device     = "cuda" if torch.cuda.is_available() else "cpu"

try:
    import psutil
    _ram_gb = psutil.virtual_memory().total / (1024**3)
except ImportError:
    _ram_gb = 16.0
PIN_MEMORY = torch.cuda.is_available() and (_ram_gb >= 8)

log.info(f"Device        : {device}")
log.info(f"RAM           : {_ram_gb:.1f} GB  →  pin_memory={PIN_MEMORY}")
log.info(f"WORK_DIR      : {WORK_DIR}")
log.info(f"SEG_DATA_DIR  : {SEG_DATA_DIR}")
log.info(f"CLASS_DATA_DIR: {CLASS_DATA_DIR}")
log.info(f"PyTorch       : {torch.__version__}")
log.info(f"IMG_SIZE      : {IMG_SIZE}")

# ── Sanity checks ──────────────────────────────────────────────────────
if not os.path.exists(SEG_DATA_DIR) or not os.listdir(SEG_DATA_DIR):
    raise RuntimeError(
        "❌ Segmentation dataset not found!\n"
        f"   Expected: {SEG_DATA_DIR}\n"
        "   → Run Cell 0 first."
    )

if not os.path.exists(CLASS_DATA_DIR) or not os.listdir(CLASS_DATA_DIR):
    raise RuntimeError(
        "❌ Classification dataset not found!\n"
        f"   Expected: {CLASS_DATA_DIR}\n"
        "   → Run Cell 0 first."
    )

log.info("Both datasets OK ✅")


### Cell 1.2 — Build Glyph2025 from Dataset

In [ ]:
import cv2

def _find_class_root(base_dir: str) -> str:
    """Walk until we find a folder whose children match Gardiner code pattern."""
    pattern = re.compile(r"^[A-Za-z]{1,3}\d+$")
    for root, dirs, _ in os.walk(base_dir):
        if sum(1 for d in dirs if pattern.match(d)) >= 5:
            return root
    return base_dir

def _safe_copy(src_path: str, dst_dir: str) -> str:
    fname = os.path.basename(src_path).lower()
    dst   = os.path.join(dst_dir, fname)
    if os.path.exists(dst):
        stem, ext = os.path.splitext(fname)
        n = 1
        while os.path.exists(os.path.join(dst_dir, f"{stem}_{n}{ext}")):
            n += 1
        dst = os.path.join(dst_dir, f"{stem}_{n}{ext}")
    try:
        shutil.copy2(src_path, dst)
    except Exception as e:
        log.warning(f"Copy skip {src_path}: {e}")
    return dst

def _merge_class_folders(sources: list, dst: str) -> int:
    """Copy class-folder images into GLYPH2025_RAW."""
    os.makedirs(dst, exist_ok=True); total = 0
    for src in sources:
        if not os.path.exists(src):
            log.warning(f"Source not found: {src}"); continue
        for cls in os.listdir(src):
            cls_src = os.path.join(src, cls)
            if not os.path.isdir(cls_src): continue
            cls_dst = os.path.join(dst, cls.lower())
            os.makedirs(cls_dst, exist_ok=True)
            for f in os.listdir(cls_src):
                if f.lower().endswith(IMG_EXTS):
                    _safe_copy(os.path.join(cls_src, f), cls_dst)
                    total += 1
    return total

# ── Build GLYPH2025_RAW from Classification dataset (GitHub) ──────────
if not os.path.exists(GLYPH2025_RAW):
    log.info("Building GLYPH2025_RAW from classification dataset ...")

    # الـ GitHub repo: rfuentesfe/EgyptianHieroglyphicText
    # الـ structure: repo/dataset/<class_name>/<images>
    hiero_dir = os.path.join(CLASS_DATA_DIR, "EgyptianHieroglyphicText")

    if not os.path.exists(hiero_dir):
        raise RuntimeError(
            f"❌ Classification dataset not found at {hiero_dir}\n"
            "   → Run Cell 0 first to clone the GitHub repo."
        )

    # ابحث عن الـ class folders جوا الـ repo
    class_root = _find_class_root(hiero_dir)
    log.info(f"Class root detected: {class_root}")

    # تحقق إن فيه Gardiner folders فعلاً
    _pattern  = re.compile(r"^[A-Za-z]{1,3}\d+$")
    gly_dirs  = [d for d in os.listdir(class_root)
                 if os.path.isdir(os.path.join(class_root, d))
                 and _pattern.match(d)]

    log.info(f"Gardiner class folders found: {len(gly_dirs)}")
    if gly_dirs:
        log.info(f"Sample classes: {sorted(gly_dirs)[:10]}")

    if len(gly_dirs) < 5:
        # fallback: بدور في كل subfolders
        log.info("Searching deeper for class folders ...")
        for root, dirs, _ in os.walk(hiero_dir):
            gly = [d for d in dirs if _pattern.match(d)]
            if len(gly) >= 5:
                class_root = root
                gly_dirs   = gly
                log.info(f"Found at: {class_root} | {len(gly_dirs)} classes")
                break

    if len(gly_dirs) < 5:
        raise RuntimeError(
            f"❌ Could not find Gardiner class folders in {hiero_dir}\n"
            f"   Repo structure: {os.listdir(hiero_dir)[:10]}\n"
            "   Expected folders like A1, B1, G17 ..."
        )

    n = _merge_class_folders([class_root], GLYPH2025_RAW)
    log.info(f"✅ GLYPH2025_RAW built — {n:,} images from {len(gly_dirs)} classes")

else:
    n_cls = len([d for d in os.listdir(GLYPH2025_RAW)
                 if os.path.isdir(os.path.join(GLYPH2025_RAW, d))])
    n_img = sum(len(os.listdir(os.path.join(GLYPH2025_RAW, d)))
                for d in os.listdir(GLYPH2025_RAW)
                if os.path.isdir(os.path.join(GLYPH2025_RAW, d)))
    log.info(f"GLYPH2025_RAW already exists → {n_cls} classes | {n_img:,} images")


### Cell 1.3 — Resize & Filter

In [ ]:
def resize_and_pad(src_path: str, dst_path: str, size: int = IMG_SIZE):
    """Letterbox-resize preserving aspect ratio with black padding."""
    try:
        img = Image.open(src_path).convert("RGB")
        img.thumbnail((size, size), Image.LANCZOS)
        canvas = Image.new("RGB", (size, size), (0, 0, 0))
        canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        canvas.save(dst_path)
    except Exception as e:
        log.warning(f"Resize skip {src_path}: {e}")

if not os.path.exists(GLYPH2025_DIR):
    # Guard: make sure GLYPH2025_RAW is non-empty before trying to list it
    if not os.path.exists(GLYPH2025_RAW) or not os.listdir(GLYPH2025_RAW):
        raise RuntimeError(
            "❌ GLYPH2025_RAW is empty or missing!\n"
            "   → Run Cell 1.2 first to build the dataset."
        )

    log.info("Resizing images ...")
    os.makedirs(GLYPH2025_DIR, exist_ok=True)
    pairs = [
        (os.path.join(GLYPH2025_RAW, cls, f),
         os.path.join(GLYPH2025_DIR, cls, f))
        for cls in os.listdir(GLYPH2025_RAW)
        if os.path.isdir(os.path.join(GLYPH2025_RAW, cls))
        for f in os.listdir(os.path.join(GLYPH2025_RAW, cls))
        if f.lower().endswith(IMG_EXTS)
    ]
    if not pairs:
        raise RuntimeError(
            "❌ No images found in GLYPH2025_RAW!\n"
            "   → Re-run Cell 1.2 — the dataset may not have extracted correctly."
        )
    for src, dst in tqdm(pairs, desc="Resizing"):
        resize_and_pad(src, dst)
    log.info(f"{len(pairs):,} images resized to {IMG_SIZE}×{IMG_SIZE}")
else:
    n_cls = len([d for d in os.listdir(GLYPH2025_DIR)
                 if os.path.isdir(os.path.join(GLYPH2025_DIR, d))])
    n_img = sum(len(os.listdir(os.path.join(GLYPH2025_DIR, d)))
                for d in os.listdir(GLYPH2025_DIR)
                if os.path.isdir(os.path.join(GLYPH2025_DIR, d)))
    log.info(f"Glyph2025_processed already exists → {n_cls} classes | {n_img:,} images")

# ── 164-class Gardiner filter ──────────────────────────────────────────────
OFFICIAL_CLASSES = {
    "a1","a2","a19","a24","a30","a40","a42","a50","b1",
    "d1","d2","d4","d21","d28","d35","d36","d37","d39","d40",
    "d45","d46","d52","d54","d55","d56","d58","d60",
    "e1","e7","e21","e23","e34",
    "f1","f4","f12","f13","f18","f21","f26","f31","f32","f34","f35","f39",
    "g1","g5","g7","g14","g17","g25","g35","g36","g37","g38","g39","g40","g43",
    "h1","h6","h8","i1","i9","i10","l1","l2",
    "m2","m3","m12","m16","m17","m18","m20","m22","m23","m42",
    "n1","n5","n8","n14","n18","n25","n26","n29","n30","n31",
    "n33","n35","n36","n37","n42",
    "o1","o3","o4","o6","o28","o29","o34","o49","o50","p8",
    "q1","q2","q3","r4","r7","r8","r11","r14",
    "s3","s19","s21","s24","s27","s28","s29","s34","s38","s40","s42",
    "t21","t22","t28","u1","u6","u7","u15","u23","u33",
    "v1","v4","v6","v7","v10","v13","v20","v28","v29","v30","v31",
    "w11","w14","w15","w17","w18","w19","w22","w23","w24","w25",
    "x1","x7","x8","y1","y2","y3","y4","y5",
    "z1","z2","z3","z4","z11","aa13","aa15",
}
MIN_SAMPLES = 7

class_counts = {
    cls: sum(1 for f in os.listdir(os.path.join(GLYPH2025_DIR, cls))
             if f.lower().endswith(IMG_EXTS))
    for cls in os.listdir(GLYPH2025_DIR)
    if os.path.isdir(os.path.join(GLYPH2025_DIR, cls))
}
final_classes = sorted(c for c in class_counts
                       if class_counts[c] >= MIN_SAMPLES and c in OFFICIAL_CLASSES)

if not final_classes:
    log.error("❌ No Gardiner classes found after filtering!")
    log.info(f"   All detected classes: {sorted(class_counts.keys())[:20]}")
    log.info(f"   Total classes in processed dir: {len(class_counts)}")
    log.info("   Possible causes:")
    log.info("   1. Dataset has no Gardiner-code category names")
    log.info("   2. All classes have < 7 images (MIN_SAMPLES too high?)")
    raise RuntimeError("No valid Gardiner classes found — check log above.")

rows = [(os.path.join(GLYPH2025_DIR, cls, f), cls)
        for cls in final_classes
        for f in os.listdir(os.path.join(GLYPH2025_DIR, cls))
        if f.lower().endswith(IMG_EXTS)]

df           = pd.DataFrame(rows, columns=["path", "label"])
classes      = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
df["y"]      = df["label"].map(class_to_idx)

with open(STEP1_STATE, "w") as f:
    json.dump({"classes": list(classes), "class_to_idx": class_to_idx,
               "n_images": len(df)}, f, indent=2)

log.info(f"Classes: {len(classes)} | Images: {len(df):,}")
log.info("Step 1 complete ✅")


### Cell 1.4 — 📊 Step 1 Visualisation

In [ ]:
counts = df['label'].value_counts().sort_index()
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Step 1 — Dataset Overview', fontsize=15, fontweight='bold')

# ── Distribution ──────────────────────────────────────────────────────────
axes[0].bar(range(len(counts)), sorted(counts.values, reverse=True),
            color='steelblue', edgecolor='none', width=1.0)
axes[0].axhline(counts.mean(), color='tomato', ls='--', lw=1.5,
                label=f'mean = {counts.mean():.0f}')
axes[0].set_title(f'Class Distribution ({len(classes)} classes)')
axes[0].set_xlabel('Class rank'); axes[0].set_ylabel('# images')
axes[0].legend()

# ── Top/Bottom 20 ─────────────────────────────────────────────────────────
top20 = counts.nlargest(20); bot20 = counts.nsmallest(20)
axes[1].barh(top20.index[::-1], top20.values[::-1], color='seagreen')
axes[1].set_title('Top-20 Classes'); axes[1].set_xlabel('# images')
axes[2].barh(bot20.index[::-1], bot20.values[::-1], color='tomato')
axes[2].set_title('Bottom-20 Classes'); axes[2].set_xlabel('# images')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'step1_overview.png'), dpi=120)
plt.show()

# ── Sample grid ───────────────────────────────────────────────────────────
sample_cls = random.sample(list(classes), min(16, len(classes)))
fig2, axes2 = plt.subplots(4, 4, figsize=(10, 10))
for ax, cls in zip(axes2.flatten(), sample_cls):
    imgs = df[df['label'] == cls]['path'].tolist()
    try:
        ax.imshow(Image.open(random.choice(imgs)).convert('RGB'))
        ax.set_title(cls, fontsize=9)
    except Exception:
        pass
    ax.axis('off')
plt.suptitle('Random Glyph Samples', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
log.info('Step 1 visualisation done')

### ⚡ ENDPOINT 1 — Resume from Step 1

In [ ]:
if not os.path.exists(STEP1_STATE):
    raise FileNotFoundError('Run Cells 1.1–1.3 first.')
with open(STEP1_STATE) as f: _s1 = json.load(f)
classes      = _s1['classes']
class_to_idx = _s1['class_to_idx']
idx_to_class = {int(i): c for c, i in class_to_idx.items()}
rows = [(os.path.join(GLYPH2025_DIR, cls, fn), cls)
        for cls in classes
        for fn in os.listdir(os.path.join(GLYPH2025_DIR, cls))
        if fn.lower().endswith(IMG_EXTS)]
df = pd.DataFrame(rows, columns=['path', 'label'])
df['y'] = df['label'].map(class_to_idx)
log.info(f'⚡ ENDPOINT 1 | {len(classes)} classes | {len(df):,} images')

---
# 🎯 STEP 2 — SAM Fine-tuning
> Fine-tune mask decoder on per-instance hieroglyph masks · BCE+Dice · LR warmup

### Cell 2.1 — SAM Config (validated)

In [ ]:
from segment_anything import sam_model_registry
from segment_anything.utils.transforms import ResizeLongestSide

@dataclass
class SAMFineTuneCFG:
    model_type            : str   = 'vit_b'
    sam_checkpoint        : str   = SAM_CKPT
    save_dir              : str   = SAM_FINETUNE_DIR
    epochs                : int   = 20
    batch_size            : int   = 2
    lr                    : float = 1e-4
    weight_decay          : float = 1e-4
    num_workers           : int   = 2
    grad_accum_steps      : int   = 2
    mixed_precision       : bool  = True
    val_ratio             : float = 0.15
    bbox_jitter           : int   = 8
    min_mask_area         : int   = 16
    sam_image_size        : int   = 1024
    train_decoder_only    : bool  = True
    unfreeze_last_n_blocks: int   = 0
    bce_weight            : float = 1.0
    dice_weight           : float = 1.0
    pred_threshold        : float = 0.5
    warmup_epochs         : int   = 2

    def __post_init__(self):
        assert self.epochs > 0,                      'epochs must be > 0'
        assert self.batch_size > 0,                  'batch_size must be > 0'
        assert 0 < self.lr < 1,                      'lr must be in (0, 1)'
        assert 0 <= self.val_ratio < 1,              'val_ratio must be in [0, 1)'
        assert self.warmup_epochs < self.epochs,     'warmup_epochs must be < epochs'
        assert 0 <= self.pred_threshold <= 1,        'pred_threshold must be in [0, 1]'

sam_cfg = SAMFineTuneCFG()
os.makedirs(sam_cfg.save_dir, exist_ok=True)
log.info(f'SAMFineTuneCFG validated ✅  lr={sam_cfg.lr}  epochs={sam_cfg.epochs}  warmup={sam_cfg.warmup_epochs}')

### Cell 2.2 — Dataset Preparation (per-instance masks)

In [ ]:
# ── Download SAM checkpoint ───────────────────────────────────────────────
if not os.path.exists(sam_cfg.sam_checkpoint):
    url_map = {
        'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
        'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
    }
    log.info(f'Downloading SAM {sam_cfg.model_type} checkpoint ...')
    subprocess.run(['wget', '-q', url_map[sam_cfg.model_type],
                    '-O', sam_cfg.sam_checkpoint], check=True)
log.info(f'SAM checkpoint ready: {sam_cfg.sam_checkpoint}')

# ── FIX: per-instance masks — one (image, mask) pair per annotation ───────
def coco_to_instance_masks(src_root, out_root, json_name, split_name):
    """
    Save one binary mask per annotation instance (not per image).
    This gives SAM correct per-glyph supervision.
    """
    src_root = Path(src_root); out_root = Path(out_root)
    json_path = src_root / json_name
    if not json_path.exists():
        raise FileNotFoundError(f'COCO JSON not found: {json_path}')

    (out_root / split_name / 'images').mkdir(parents=True, exist_ok=True)
    (out_root / split_name / 'masks').mkdir(parents=True, exist_ok=True)
    out_img  = out_root / split_name / 'images'
    out_mask = out_root / split_name / 'masks'

    with open(json_path) as f: data = json.load(f)
    img_meta = {img['id']: img for img in data['images']}
    written  = 0

    for ann in tqdm(data['annotations'], desc=f'Prep {split_name}'):
        img_info = img_meta.get(ann['image_id'])
        if img_info is None: continue
        src_path = src_root / img_info['file_name']
        if not src_path.exists(): continue

        h, w = int(img_info['height']), int(img_info['width'])
        mask = np.zeros((h, w), dtype=np.uint8)
        for seg in ann.get('segmentation', []):
            if not isinstance(seg, list) or len(seg) < 6: continue
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2)
            cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 255)

        if mask.sum() < sam_cfg.min_mask_area: continue

        stem     = f"{src_path.stem}_ann{ann['id']}"
        dst_img  = out_img  / (stem + src_path.suffix)
        dst_mask = out_mask / (stem + '.png')

        if not dst_img.exists():
            try: shutil.copy2(src_path, dst_img)
            except Exception as e: log.warning(f'img copy fail: {e}'); continue

        cv2.imwrite(str(dst_mask), mask)
        written += 1

    log.info(f'Prepared {written} instance pairs → {split_name}')
    return written

SAM_PREPARED = os.path.join(WORK_DIR, 'sam_data_prepared')

def _list_images(folder):
    p = Path(folder)
    if not p.exists(): return []
    return sorted(f for f in p.iterdir()
                  if f.is_file() and f.suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'})

def _mask_for(mask_dir, img_path):
    stem = Path(img_path).stem
    for ext in ['.png','.jpg','.jpeg','.bmp']:
        c = Path(mask_dir) / (stem + ext)
        if c.exists(): return str(c)
    return None

def build_sam_pairs(prepared_root):
    pairs = []
    for split in ['train', 'val']:
        img_dir  = os.path.join(prepared_root, split, 'images')
        mask_dir = os.path.join(prepared_root, split, 'masks')
        if not os.path.exists(img_dir): continue
        for img_p in _list_images(img_dir):
            mp = _mask_for(mask_dir, img_p)
            if mp: pairs.append((str(img_p), mp))
    return pairs

if not os.path.exists(SAM_PREPARED):
    log.info('Preparing per-instance SAM dataset ...')
    # Try standard Kaggle dataset JSON names
    for train_json_name in ['train.json', 'train/_annotations.coco.json']:
        train_json = os.path.join(SEG_DATA_DIR, train_json_name)
        if os.path.exists(train_json):
            coco_to_instance_masks(SEG_DATA_DIR, SAM_PREPARED, train_json_name, 'train')
            break
    for val_json_name in ['Validation.json', 'valid/_annotations.coco.json', 'val/_annotations.coco.json']:
        val_json = os.path.join(SEG_DATA_DIR, val_json_name)
        if os.path.exists(val_json):
            coco_to_instance_masks(SEG_DATA_DIR, SAM_PREPARED, val_json_name, 'val')
            break
else:
    log.info('SAM dataset already prepared')

all_sam_pairs = build_sam_pairs(SAM_PREPARED)
if not all_sam_pairs:
    log.warning('No SAM pairs found — using base SAM for segmentation')
    sam_pairs_train, sam_pairs_val = [], []
else:
    sam_pairs_train, sam_pairs_val = train_test_split(
        all_sam_pairs, test_size=sam_cfg.val_ratio, random_state=SEED, shuffle=True)
    log.info(f'SAM pairs — Train: {len(sam_pairs_train)} | Val: {len(sam_pairs_val)}')

### Cell 2.3 — Dataset, Loss & Model

In [ ]:
def dice_loss(logits, targets, smooth=1.0):
    probs   = torch.sigmoid(logits).flatten(1)
    targets = targets.flatten(1)
    inter   = (probs * targets).sum(1)
    union   = probs.sum(1) + targets.sum(1)
    return 1.0 - ((2.0 * inter + smooth) / (union + smooth)).mean()

def compute_iou_and_dice(logits, targets, threshold=0.5, eps=1e-7):
    preds   = (torch.sigmoid(logits) > threshold).float().flatten(1)
    targets = targets.flatten(1)
    inter   = (preds * targets).sum(1)
    union   = preds.sum(1) + targets.sum(1) - inter
    iou     = ((inter + eps) / (union + eps)).mean().item()
    dice    = ((2*inter + eps) / (preds.sum(1) + targets.sum(1) + eps)).mean().item()
    return iou, dice

# ── FIX 2: image loaded ONCE per __getitem__, no double-read ──────────────
class HieroglyphSegDataset(Dataset):
    def __init__(self, pairs, image_size=1024, bbox_jitter=8, min_mask_area=16):
        self.pairs         = pairs
        self.image_size    = image_size
        self.bbox_jitter   = bbox_jitter
        self.min_mask_area = min_mask_area
        self.transform     = ResizeLongestSide(image_size)

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        # Single read — no fallback that re-reads
        try:
            raw = cv2.imread(img_path)
            if raw is None: raise ValueError(f'cv2 returned None for {img_path}')
            image = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            raw_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = (raw_mask > 127).astype(np.float32) if raw_mask is not None                    else np.zeros(image.shape[:2], dtype=np.float32)
        except Exception as e:
            log.warning(f'Bad sample [{idx}] {img_path}: {e}')
            # Return a valid zero sample rather than reading again
            image = np.zeros((256, 256, 3), dtype=np.uint8)
            mask  = np.zeros((256, 256),    dtype=np.float32)

        input_image = self.transform.apply_image(image)
        input_image = torch.as_tensor(input_image).permute(2, 0, 1).float()
        h, w = image.shape[:2]
        ys, xs = np.where(mask > 0)
        if len(xs) < self.min_mask_area:
            box = np.array([0, 0, w, h], dtype=np.float32)
        else:
            j   = self.bbox_jitter
            box = np.array([max(0,int(xs.min())-j), max(0,int(ys.min())-j),
                             min(w,int(xs.max())+j), min(h,int(ys.max())+j)],
                           dtype=np.float32)
        box_t  = self.transform.apply_boxes(box[None], (h, w))
        mask_t = torch.as_tensor(mask).unsqueeze(0)
        return {'image': input_image, 'mask': mask_t,
                'box': torch.as_tensor(box_t[0]),
                'original_size': torch.tensor([h, w]),
                'image_path': img_path}

def sam_collate(batch):
    return {'images':         torch.stack([b['image'] for b in batch]),
            'masks':          [b['mask']          for b in batch],
            'boxes':          torch.stack([b['box']    for b in batch]),
            'original_sizes': torch.stack([b['original_size'] for b in batch]),
            'image_paths':    [b['image_path']    for b in batch]}

def build_sam_model(cfg: SAMFineTuneCFG):
    sam = sam_model_registry[cfg.model_type](checkpoint=cfg.sam_checkpoint)
    sam.to(device).train()
    for p in sam.parameters(): p.requires_grad = False
    for p in sam.mask_decoder.parameters(): p.requires_grad = True
    if not cfg.train_decoder_only and cfg.unfreeze_last_n_blocks > 0:
        for blk in sam.image_encoder.blocks[-cfg.unfreeze_last_n_blocks:]:
            for p in blk.parameters(): p.requires_grad = True
    trainable = sum(p.numel() for p in sam.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in sam.parameters())
    log.info(f'SAM: trainable {trainable:,} / {total:,} params')
    return sam

log.info('SAM utilities ready ✅')

### Cell 2.4 — Training Loop

In [ ]:
def forward_sam_batch(sam_model, images, boxes, original_sizes):
    """Image encoder: true batch (GPU parallel). Decoder: per-sample (required)."""
    images = images.to(device)
    boxes  = boxes.to(device)
    # Batch encode — all images in parallel on GPU
    processed  = torch.stack([sam_model.preprocess(img) for img in images])
    embeddings = sam_model.image_encoder(processed)
    pe         = sam_model.prompt_encoder.get_dense_pe()   # compute once, reuse
    outputs    = []
    for i in range(images.shape[0]):
        sparse_emb, dense_emb = sam_model.prompt_encoder(
            points=None, boxes=boxes[i].unsqueeze(0), masks=None)
        low_res, _ = sam_model.mask_decoder(
            image_embeddings         = embeddings[i].unsqueeze(0),
            image_pe                 = pe,
            sparse_prompt_embeddings = sparse_emb,
            dense_prompt_embeddings  = dense_emb,
            multimask_output         = False,
        )
        upscaled = sam_model.postprocess_masks(
            low_res,
            input_size    = images[i].shape[-2:],
            original_size = tuple(original_sizes[i].tolist()),
        )
        outputs.append(upscaled)
    return outputs

def _run_epoch(model, loader, optimizer, scaler, cfg, train=True):
    model.train() if train else model.eval()
    bce_fn  = nn.BCEWithLogitsLoss()
    running = {'loss': 0.0, 'iou': 0.0, 'dice': 0.0}
    steps   = 0
    ctx = autocast(device_type=device, enabled=(cfg.mixed_precision and device=='cuda'))
    if train: optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc='Train' if train else 'Val  ', leave=False)
    for step, batch in enumerate(pbar):
        gt_masks = [m.to(device) for m in batch['masks']]
        with (ctx if train else torch.inference_mode()):  # FIX 5: inference_mode in val
            preds = forward_sam_batch(model, batch['images'],
                                      batch['boxes'], batch['original_sizes'])
            total_loss = b_iou = b_dice = 0.0
            for pred, gt in zip(preds, gt_masks):
                gt = gt.unsqueeze(0) if gt.dim() == 3 else gt
                loss = cfg.bce_weight*bce_fn(pred,gt) + cfg.dice_weight*dice_loss(pred,gt)
                total_loss += loss
                iou, dice = compute_iou_and_dice(pred, gt, cfg.pred_threshold)
                b_iou += iou; b_dice += dice
            total_loss /= len(preds); b_iou /= len(preds); b_dice /= len(preds)

        if train:
            scaler.scale(total_loss / cfg.grad_accum_steps).backward()
            if (step + 1) % cfg.grad_accum_steps == 0:
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True)

        running['loss'] += total_loss.item()
        running['iou']  += b_iou
        running['dice'] += b_dice
        steps += 1
        pbar.set_postfix({k: f'{v/steps:.4f}' for k, v in running.items()})
    return {k: v / max(steps, 1) for k, v in running.items()}

log.info('SAM training loop ready ✅')

### Cell 2.5 — Run SAM Fine-tuning

In [ ]:
sam_finetuned = None

if not sam_pairs_train:
    log.warning('No training pairs — skipping SAM fine-tuning. Using base SAM.')

elif os.path.exists(SAM_BEST_PATH):
    log.info(f'Checkpoint exists → {SAM_BEST_PATH}')
    sam_finetuned = SAM_BEST_PATH

else:
    log.info(f'Starting SAM fine-tuning ({sam_cfg.epochs} epochs) ...')
    sam_model = build_sam_model(sam_cfg)
    train_ds  = HieroglyphSegDataset(sam_pairs_train, sam_cfg.sam_image_size,
                                      sam_cfg.bbox_jitter, sam_cfg.min_mask_area)
    val_ds    = HieroglyphSegDataset(sam_pairs_val, sam_cfg.sam_image_size,
                                      0, sam_cfg.min_mask_area)
    train_ldr = DataLoader(train_ds, sam_cfg.batch_size, shuffle=True,
                            num_workers=sam_cfg.num_workers, pin_memory=PIN_MEMORY,
                            collate_fn=sam_collate)
    val_ldr   = DataLoader(val_ds, sam_cfg.batch_size, shuffle=False,
                            num_workers=sam_cfg.num_workers, pin_memory=PIN_MEMORY,
                            collate_fn=sam_collate)
    optimizer = torch.optim.AdamW(
        [p for p in sam_model.parameters() if p.requires_grad],
        lr=sam_cfg.lr, weight_decay=sam_cfg.weight_decay)

    # FIX: LR warmup then cosine decay
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=sam_cfg.warmup_epochs)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=sam_cfg.epochs - sam_cfg.warmup_epochs, eta_min=1e-6)
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, [warmup_sched, cosine_sched], milestones=[sam_cfg.warmup_epochs])

    scaler   = GradScaler(device) if device=='cuda' else GradScaler('cpu')
    history  = []; best_iou = -1.0

    for epoch in range(1, sam_cfg.epochs + 1):
        t0 = time.time()
        tr = _run_epoch(sam_model, train_ldr, optimizer, scaler, sam_cfg, train=True)
        vl = _run_epoch(sam_model, val_ldr,   optimizer, scaler, sam_cfg, train=False)
        scheduler.step()
        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = (time.time() - t0) / 60
        log.info(f'Ep {epoch:02d} | loss={tr["loss"]:.4f}/{vl["loss"]:.4f} | '
                 f'iou={tr["iou"]:.4f}/{vl["iou"]:.4f} | '
                 f'lr={lr_now:.2e} | {elapsed:.1f}min')
        history.append({'epoch': epoch,
                        **{f'tr_{k}': v for k, v in tr.items()},
                        **{f'val_{k}': v for k, v in vl.items()},
                        'lr': lr_now})
        ckpt = {'model_state_dict': sam_model.state_dict(),
                'history': history, 'epoch': epoch}
        torch.save(ckpt, SAM_LAST_PATH)
        if vl['iou'] > best_iou:
            best_iou = vl['iou']
            torch.save(ckpt, SAM_BEST_PATH)
            log.info(f'  ✅ New best (val_iou={best_iou:.4f})')

    sam_finetuned = SAM_BEST_PATH
    log.info(f'SAM fine-tuning done ✅  best_iou={best_iou:.4f}')
    with open(STEP2_STATE, 'w') as f:
        json.dump({'sam_best_path': SAM_BEST_PATH, 'history': history}, f, indent=2)
    del sam_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

### Cell 2.6 — 📊 Step 2 Visualisation

In [ ]:
if os.path.exists(SAM_BEST_PATH):
    ckpt    = torch.load(SAM_BEST_PATH, map_location='cpu')
    history = ckpt.get('history', [])
    if history:
        ep_x    = [h['epoch']    for h in history]
        metrics = [('loss','Loss'),('iou','IoU'),('dice','Dice')]
        fig, axes = plt.subplots(1, 4, figsize=(24, 4))
        fig.suptitle('Step 2 — SAM Fine-tuning', fontsize=14, fontweight='bold')
        for ax,(key,title) in zip(axes[:3], metrics):
            ax.plot(ep_x, [h[f'tr_{key}']  for h in history], label='train', color='steelblue')
            ax.plot(ep_x, [h[f'val_{key}'] for h in history], label='val',   color='tomato')
            ax.set_title(f'SAM — {title}'); ax.legend(); ax.set_xlabel('Epoch')
        axes[3].plot(ep_x, [h.get('lr',0) for h in history], color='purple', marker='o', ms=3)
        axes[3].set_title('LR Schedule'); axes[3].set_yscale('log'); axes[3].set_xlabel('Epoch')
        plt.tight_layout()
        plt.savefig(os.path.join(WORK_DIR,'step2_sam_curves.png'), dpi=120); plt.show()
    else:
        log.info('No training history (SAM already existed or was skipped)')
else:
    log.info('SAM checkpoint not found — nothing to plot')

### ⚡ ENDPOINT 2 — Resume from Step 2

In [ ]:
if not os.path.exists(STEP1_STATE): raise FileNotFoundError('Run ENDPOINT 1 first.')
with open(STEP1_STATE) as f: _s1 = json.load(f)
classes = _s1['classes']; class_to_idx = _s1['class_to_idx']
idx_to_class = {int(i): c for c, i in class_to_idx.items()}
rows = [(os.path.join(GLYPH2025_DIR, cls, fn), cls)
        for cls in classes for fn in os.listdir(os.path.join(GLYPH2025_DIR, cls))
        if fn.lower().endswith(IMG_EXTS)]
df = pd.DataFrame(rows, columns=['path','label']); df['y'] = df['label'].map(class_to_idx)
sam_finetuned = SAM_BEST_PATH if os.path.exists(SAM_BEST_PATH) else None
log.info(f'⚡ ENDPOINT 2 | SAM={"fine-tuned" if sam_finetuned else "base"} | classes={len(classes)}')

---
# 🔍 STEP 3 — Segmentation (IGSM + Fine-tuned SAM)
> Border removal → multi-threshold → vectorised NMS → RTL reading order

### Cell 3.1 — Segmentation Config (validated)

In [ ]:
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from segment_anything import SamPredictor, SamAutomaticMaskGenerator
import glob

@dataclass
class SegConfig:
    sam_checkpoint         : str   = SAM_CKPT
    seg_output_dir         : str   = SEG_OUTPUT_DIR
    border_thresh          : int   = 35
    denoise_h              : int   = 10
    denoise_template       : int   = 7
    denoise_search         : int   = 21
    clahe_clip             : float = 2.5
    clahe_grid             : tuple = field(default_factory=lambda: (8,8))
    adaptive_block         : int   = 25
    adaptive_c             : int   = 8
    min_area               : int   = 80
    stroke_min_aspect      : float = 3.5
    stroke_min_width       : int   = 15
    stroke_max_height      : int   = 40
    stroke_min_area        : int   = 60
    stroke_min_solidity    : float = 0.35
    stroke_group_gap       : int   = 90
    parallel_x_overlap     : float = 0.60
    min_mask_width_ratio   : float = 0.03
    min_mask_height_ratio  : float = 0.04
    noise_min_density      : float = 0.18
    max_area_ratio         : float = 0.40
    points_per_side        : int   = 32
    pred_iou_thresh        : float = 0.82
    stability_score_thresh : float = 0.90
    min_mask_region_area   : int   = 80
    box_nms_thresh         : float = 0.50
    min_solidity           : float = 0.30
    max_aspect_ratio       : float = 8.0
    iou_threshold          : float = 0.35
    padding                : int   = 6
    bg_fill                : str   = 'white'
    output_size            : tuple = field(default_factory=lambda: (128,128))

    def __post_init__(self):
        assert 0 < self.pred_iou_thresh  <= 1
        assert 0 < self.stability_score_thresh <= 1
        assert 0 < self.iou_threshold    <= 1
        assert 0 < self.max_area_ratio   <= 1
        assert self.min_area > 0
        assert self.adaptive_block % 2 == 1, 'adaptive_block must be odd'

seg_cfg = SegConfig()
if sam_finetuned and os.path.exists(sam_finetuned):
    seg_cfg.sam_checkpoint = sam_finetuned
    log.info(f'Using fine-tuned SAM: {sam_finetuned}')
else:
    log.info(f'Using base SAM: {seg_cfg.sam_checkpoint}')
log.info('SegConfig validated ✅')

### Cell 3.2 — Load SAM

In [ ]:
def load_sam_auto(cfg: SegConfig):
    if not os.path.exists(cfg.sam_checkpoint):
        log.info('Downloading SAM checkpoint ...')
        subprocess.run(['wget','-q',
            'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
            '-O', SAM_CKPT], check=True)
        cfg.sam_checkpoint = SAM_CKPT
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    raw = torch.load(cfg.sam_checkpoint, map_location=dev)
    if isinstance(raw, dict) and 'model_state_dict' in raw:
        sam = sam_model_registry['vit_b'](checkpoint=SAM_CKPT)
        sam.load_state_dict(raw['model_state_dict'])
    else:
        sam = sam_model_registry['vit_b'](checkpoint=cfg.sam_checkpoint)
    sam.to(device=dev)
    return SamAutomaticMaskGenerator(
        model=sam,
        points_per_side        = cfg.points_per_side,
        pred_iou_thresh        = cfg.pred_iou_thresh,
        stability_score_thresh = cfg.stability_score_thresh,
        min_mask_region_area   = cfg.min_mask_region_area,
        box_nms_thresh         = cfg.box_nms_thresh,
    )

sam_generator = load_sam_auto(seg_cfg)
log.info('SAM generator ready ✅')

### Cell 3.3 — IGSM Pipeline Functions

In [ ]:
def load_image(path: str) -> np.ndarray:
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(f'Cannot read: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _detect_glyph_color(image):
    hsv  = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    dark = hsv[:,:,2] < 100
    if dark.sum() == 0: return 'dark'
    return 'color' if hsv[:,:,1][dark].mean() > 40 else 'dark'

def remove_border(image, cfg):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV); sat = hsv[:,:,1]
    H, W = image.shape[:2]; margin = max(8, min(H,W)//20)
    edges = {k: v.mean() for k,v in [
        ('top',sat[:margin,:]),('bottom',sat[-margin:,:]),
        ('left',sat[:,:margin]),('right',sat[:,-margin:])]}
    high = {k for k,v in edges.items() if v > cfg.border_thresh}
    if not high: return image
    y1 = margin if 'top'    in high else 0
    y2 = H-margin if 'bottom' in high else H
    x1 = margin if 'left'   in high else 0
    x2 = W-margin if 'right'  in high else W
    c = image[y1:y2, x1:x2]
    return c if c.shape[0]>=30 and c.shape[1]>=30 else image

def preprocess(image, cfg):
    gray     = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    clean    = cv2.fastNlMeansDenoising(gray, None, cfg.denoise_h,
                                         cfg.denoise_template, cfg.denoise_search)
    clahe    = cv2.createCLAHE(clipLimit=cfg.clahe_clip, tileGridSize=cfg.clahe_grid)
    enhanced = clahe.apply(clean)
    _, otsu  = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    if _detect_glyph_color(image) == 'color':
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        combined = cv2.bitwise_or(
            cv2.bitwise_and((hsv[:,:,1]>50).astype(np.uint8)*255,
                            (hsv[:,:,2]<160).astype(np.uint8)*255), otsu)
    else:
        ag = cv2.adaptiveThreshold(enhanced,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV,cfg.adaptive_block,cfg.adaptive_c)
        am = cv2.adaptiveThreshold(enhanced,255,cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV,cfg.adaptive_block+10,cfg.adaptive_c-2)
        combined = cv2.bitwise_or(cv2.bitwise_or(otsu,ag),am)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    return (cv2.morphologyEx(cv2.morphologyEx(combined,cv2.MORPH_CLOSE,k,iterations=1),
                              cv2.MORPH_OPEN,k,iterations=1), enhanced)

def remove_noise(binary, cfg):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary)
    out = np.zeros_like(binary)
    for i in range(1, n):
        w=stats[i,cv2.CC_STAT_WIDTH]; h=stats[i,cv2.CC_STAT_HEIGHT]; a=stats[i,cv2.CC_STAT_AREA]
        if w>10 and h>10 and a>cfg.min_area and a/max(w*h,1)>=cfg.noise_min_density:
            out[labels==i] = 255
    return out

def detect_strokes(binary, cfg):
    n,labels,stats,_ = cv2.connectedComponentsWithStats(binary)
    H,W = binary.shape; raw, sid = [], set()
    for i in range(1, n):
        x=stats[i,cv2.CC_STAT_LEFT]; y=stats[i,cv2.CC_STAT_TOP]
        w=stats[i,cv2.CC_STAT_WIDTH]; h=stats[i,cv2.CC_STAT_HEIGHT]; a=stats[i,cv2.CC_STAT_AREA]
        if not (w and h and a>=cfg.stroke_min_area): continue
        roi=(labels[y:y+h,x:x+w]==i).astype(np.uint8)*255
        cnts,_=cv2.findContours(roi,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        sol=0.0
        if cnts:
            ha=cv2.contourArea(cv2.convexHull(cnts[0]))
            if ha>0: sol=a/ha
        if (w/max(h,1)>=cfg.stroke_min_aspect and w>=cfg.stroke_min_width
                and h<=cfg.stroke_max_height and sol>=cfg.stroke_min_solidity):
            m=np.zeros(binary.shape,dtype=np.uint8); m[labels==i]=1
            raw.append({'mask':m,'x1':x,'y1':y,'x2':x+w,'y2':y+h}); sid.add(i)
    if not raw: return [], binary.copy()
    raw.sort(key=lambda s:s['y1'])
    p=list(range(len(raw)))
    def find(i):
        while p[i]!=i: p[i]=p[p[i]]; i=p[i]
        return i
    def union(i,j):
        pi,pj=find(i),find(j)
        if pi!=pj: p[pi]=pj
    for i in range(len(raw)):
        for j in range(i+1,len(raw)):
            ov=max(0,min(raw[i]['x2'],raw[j]['x2'])-max(raw[i]['x1'],raw[j]['x1']))
            ratio=ov/max(min(raw[i]['x2']-raw[i]['x1'],raw[j]['x2']-raw[j]['x1']),1)
            if ratio>=cfg.parallel_x_overlap: union(i,j)
    pg=defaultdict(list)
    for i,s in enumerate(raw): pg[find(i)].append(s)
    fg=[]
    for grp in pg.values():
        grp.sort(key=lambda s:s['y1']); cur=[grp[0]]
        for s in grp[1:]:
            if s['y1']-cur[-1]['y2']<=max(cfg.stroke_group_gap,int(H*0.12)): cur.append(s)
            else: fg.append(cur); cur=[s]
        fg.append(cur)
    stroke_masks=[]
    for grp in fg:
        c=np.zeros(binary.shape,dtype=np.uint8)
        for s in grp: c=np.logical_or(c,s['mask']).astype(np.uint8)
        stroke_masks.append(c)
    rem=binary.copy()
    for i in sid: rem[labels==i]=0
    return stroke_masks, rem

def _compute_solidity(mask):
    cnts,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return 0.0
    cnt=max(cnts,key=cv2.contourArea); ha=cv2.contourArea(cv2.convexHull(cnt))
    return float(cv2.contourArea(cnt)/ha) if ha>0 else 0.0

def sam_auto_segment(image, generator, cfg):
    H,W=image.shape[:2]; img_area=H*W
    min_a=img_area*0.0002; max_a=img_area*cfg.max_area_ratio
    try: raw=generator.generate(image)
    except Exception as e: log.warning(f'SAM generate failed: {e}'); return []
    filtered=[]
    for m in raw:
        mask=m['segmentation'].astype(np.uint8); area=float(mask.sum())
        if not (min_a<=area<=max_a): continue
        ys,xs=np.where(mask)
        if not len(xs): continue
        w,h=xs.max()-xs.min()+1,ys.max()-ys.min()+1
        if w/max(h,1)>cfg.max_aspect_ratio: continue
        if _compute_solidity(mask)<cfg.min_solidity: continue
        filtered.append(mask)
    return filtered

def size_filter(masks, shape, cfg):
    H,W=shape[:2]; max_a=H*W*cfg.max_area_ratio; kept=[]
    for m in masks:
        if not m.sum(): continue
        ys,xs=np.where(m==1)
        mw=int(xs.max()-xs.min()); mh=int(ys.max()-ys.min()); a=int(m.sum())
        if (a<max_a and mw>=W*cfg.min_mask_width_ratio
                and mh>=H*cfg.min_mask_height_ratio
                and a/max(mw*mh,1)>=cfg.noise_min_density):
            kept.append(m)
    return kept

# ── FIX 1: vectorised NMS — bbox IoU pre-filter, then mask IoU only when needed ──
def _masks_to_boxes(masks):
    """Extract bounding boxes from list of binary masks → (N,4) array [x1,y1,x2,y2]."""
    boxes = []
    for m in masks:
        ys, xs = np.where(m == 1)
        if len(xs) == 0: boxes.append([0,0,0,0])
        else: boxes.append([int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())])
    return np.array(boxes, dtype=np.float32)

def _bbox_iou_matrix(boxes):
    """Compute N×N IoU matrix of bounding boxes. Vectorised — no Python loop."""
    x1 = boxes[:,0]; y1 = boxes[:,1]; x2 = boxes[:,2]; y2 = boxes[:,3]
    areas = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    # Broadcast to N×N
    inter_x1 = np.maximum(x1[:,None], x1[None,:])
    inter_y1 = np.maximum(y1[:,None], y1[None,:])
    inter_x2 = np.minimum(x2[:,None], x2[None,:])
    inter_y2 = np.minimum(y2[:,None], y2[None,:])
    inter    = np.maximum(0, inter_x2-inter_x1) * np.maximum(0, inter_y2-inter_y1)
    union    = areas[:,None] + areas[None,:] - inter
    return inter / (union + 1e-6)

def _mask_iou(m1, m2):
    return float(np.logical_and(m1,m2).sum()) / (float(np.logical_or(m1,m2).sum()) + 1e-6)

def apply_nms(masks, cfg, expected_count=None):
    """
    FIX 1: O(n) bbox pre-filter + mask IoU only for candidate pairs.
    Old code: O(n²) full mask comparisons for every pair.
    New code: bbox IoU matrix once → only compute expensive mask IoU
              for pairs whose bbox IoU > threshold (typically very few).
    """
    if not masks: return []
    areas = np.array([m.sum() for m in masks], dtype=np.float32)
    boxes = _masks_to_boxes(masks)

    def _run(thr):
        # Sort by area descending (greedy NMS)
        order = np.argsort(-areas)
        bbox_ious = _bbox_iou_matrix(boxes)
        keep = []
        suppressed = np.zeros(len(masks), dtype=bool)
        for i in order:
            if suppressed[i]: continue
            keep.append(int(i))
            # Only check mask IoU for boxes with bbox overlap > threshold
            candidates = np.where((bbox_ious[i] > thr) & (~suppressed))[0]
            for j in candidates:
                if j == i: continue
                if _mask_iou(masks[i], masks[j]) > thr:
                    suppressed[j] = True
        return keep

    thr  = cfg.iou_threshold
    kept = _run(thr)
    if expected_count:
        for _ in range(20):
            if len(kept) <= expected_count: break
            thr += 0.05; kept = _run(thr)
    return [masks[i] for i in kept]

def _cluster_rows_adaptive(data, H):
    """Row tolerance = 0.5 × median glyph height, clamped to [3%, 12%] of H."""
    if not data: return []
    heights  = [d['y2']-d['y1'] for d in data]
    median_h = float(np.median(heights)) if heights else H*0.05
    tol      = max(H*0.03, min(median_h*0.5, H*0.12))
    data_s   = sorted(data, key=lambda x: x['cy'])
    rows,cur,cy = [],[data_s[0]],data_s[0]['cy']
    for item in data_s[1:]:
        if abs(item['cy']-cy)<=tol: cur.append(item)
        else: rows.append(cur); cur=[item]; cy=item['cy']
    rows.append(cur)
    return rows

def reading_order(image, masks, cfg=None):
    if cfg is None: cfg=SegConfig()
    H,W=image.shape[:2]; ann=[]
    for m in masks:
        ys,xs=np.where(m==1)
        if not len(xs): continue
        x1,y1,x2,y2=xs.min(),ys.min(),xs.max(),ys.max()
        ann.append(dict(mask=m,x1=int(x1),y1=int(y1),x2=int(x2),y2=int(y2),
                        cx=(x1+x2)/2,cy=(y1+y2)/2))
    if not ann: return []
    rows=_cluster_rows_adaptive(ann,H); ordered=[]
    if H>W:
        rows.sort(key=lambda r:-np.mean([g['cx'] for g in r]))
        for row in rows: row.sort(key=lambda g:g['cy']); ordered.extend(row)
    else:
        rows.sort(key=lambda r:np.mean([g['cy'] for g in r]))
        for row in rows: row.sort(key=lambda g:-g['cx']); ordered.extend(row)  # RTL
    return [g['mask'] for g in ordered]

def save_crops(image, masks, cfg, image_name='image'):
    os.makedirs(cfg.seg_output_dir, exist_ok=True)
    H,W=image.shape[:2]; crops=[]
    base=os.path.splitext(os.path.basename(str(image_name)))[0]
    for idx,tm in enumerate(masks):
        ys,xs=np.where(tm==1)
        if not len(xs): continue
        x1=max(0,int(xs.min())-cfg.padding); y1=max(0,int(ys.min())-cfg.padding)
        x2=min(W,int(xs.max())+cfg.padding); y2=min(H,int(ys.max())+cfg.padding)
        er=np.zeros((H,W),dtype=np.uint8); er[y1:y2,x1:x2]=1; er[tm==1]=0
        crop=image[y1:y2,x1:x2].copy(); er_r=er[y1:y2,x1:x2].astype(bool)
        if cfg.bg_fill=='white': crop[er_r]=255
        oh,ow=crop.shape[:2]; side=max(oh,ow)
        sq=np.full((side,side,3),255,dtype=np.uint8)
        sq[(side-oh)//2:(side-oh)//2+oh,(side-ow)//2:(side-ow)//2+ow]=crop
        std=cv2.resize(sq,cfg.output_size,interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(cfg.seg_output_dir,f'{base}_glyph_{idx+1:03d}.png'),
                    cv2.cvtColor(std,cv2.COLOR_RGB2BGR))
        crops.append(std)
    return crops

def run_single(image_path, cfg=None, generator=None, expected_glyphs=None, show=True):
    """Full IGSM+SAM pipeline on one stela image. Fully error-safe."""
    if cfg is None:       cfg=SegConfig()
    if generator is None: raise ValueError('Pass sam_generator!')
    try:
        image = load_image(image_path)
    except Exception as e:
        log.error(f'Cannot load {image_path}: {e}'); return [], []
    image  = remove_border(image, cfg)
    binary,_ = preprocess(image, cfg)
    binary = remove_noise(binary, cfg)
    stroke_masks,_ = detect_strokes(binary, cfg)
    sam_masks      = sam_auto_segment(image, generator, cfg)
    all_m  = size_filter(sam_masks+stroke_masks, image.shape, cfg)
    final  = apply_nms(all_m, cfg, expected_count=expected_glyphs)
    ordered= reading_order(image, final, cfg)
    crops  = save_crops(image, ordered, cfg, image_name=image_path)
    log.info(f'{os.path.basename(image_path)}: {len(ordered)} glyphs')
    if show:
        numbered=image.copy(); colors=plt.cm.tab20.colors
        for idx,m in enumerate(ordered):
            ys,xs=np.where(m==1)
            if not len(xs): continue
            x1,y1,x2,y2=xs.min(),ys.min(),xs.max(),ys.max()
            c=tuple(int(v*255) for v in colors[idx%20][:3])
            cv2.rectangle(numbered,(x1,y1),(x2,y2),c,2)
            cv2.putText(numbered,str(idx+1),(x1+2,y1+14),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
        fig,axes=plt.subplots(1,3,figsize=(18,6))
        axes[0].imshow(image);              axes[0].set_title('Original');  axes[0].axis('off')
        axes[1].imshow(binary,cmap='gray'); axes[1].set_title('Binary');    axes[1].axis('off')
        axes[2].imshow(numbered);           axes[2].set_title(f'{len(ordered)} Glyphs (RTL)'); axes[2].axis('off')
        plt.suptitle(os.path.basename(image_path), fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()
    return ordered, crops

log.info('Step 3 pipeline ready ✅')

### Cell 3.4 — 📊 Step 3 Visualisation

In [ ]:
sample_images = (glob.glob(os.path.join(SEG_DATA_DIR,'**','*.jpg'), recursive=True)
               + glob.glob(os.path.join(SEG_DATA_DIR,'**','*.png'), recursive=True))[:4]
# Prefer stela/test images if available
stela_images  = [p for p in sample_images if 'test' in p.lower() or 'stela' in p.lower()]
sample_images = (stela_images or sample_images)[:3]

seg_stats = []
for img_path in sample_images:
    try:
        ordered, crops = run_single(img_path, cfg=seg_cfg, generator=sam_generator, show=True)
        seg_stats.append({'image': os.path.basename(img_path), 'n_glyphs': len(ordered)})
    except Exception as e:
        log.warning(f'Seg failed {img_path}: {e}')

if seg_stats:
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar([s['image'] for s in seg_stats],[s['n_glyphs'] for s in seg_stats],color='steelblue')
    ax.set_title('Glyphs Detected per Image'); ax.set_ylabel('# glyphs')
    plt.tight_layout(); plt.show()

saved = glob.glob(os.path.join(SEG_OUTPUT_DIR,'*.png'))[:16]
if saved:
    fig, axes = plt.subplots(2, 8, figsize=(18,5))
    for ax,p in zip(axes.flatten(), saved):
        try: ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p)[-10:],fontsize=7)
        except: pass
        ax.axis('off')
    for ax in axes.flatten()[len(saved):]: ax.axis('off')
    plt.suptitle(f'Saved Crop Samples ({len(saved)})', fontsize=13)
    plt.tight_layout(); plt.show()
log.info('Step 3 visualisation done')

### ⚡ ENDPOINT 3 — Resume from Step 3

In [ ]:
if not os.path.exists(STEP1_STATE): raise FileNotFoundError('Run ENDPOINT 1 first.')
with open(STEP1_STATE) as f: _s1 = json.load(f)
classes = _s1['classes']; class_to_idx = _s1['class_to_idx']
idx_to_class = {int(i):c for c,i in class_to_idx.items()}
rows = [(os.path.join(GLYPH2025_DIR,cls,fn),cls)
        for cls in classes for fn in os.listdir(os.path.join(GLYPH2025_DIR,cls))
        if fn.lower().endswith(IMG_EXTS)]
df = pd.DataFrame(rows,columns=['path','label']); df['y'] = df['label'].map(class_to_idx)
sam_finetuned = SAM_BEST_PATH if os.path.exists(SAM_BEST_PATH) else None
seg_cfg       = SegConfig(sam_checkpoint=sam_finetuned or SAM_CKPT)
sam_generator = load_sam_auto(seg_cfg)
log.info(f'⚡ ENDPOINT 3 | SAM loaded | classes={len(classes)}')

---
# 🧠 STEP 4 — Classification (CVV)
> ConvNeXt-Tiny · 5-fold · MixUp · 7-view TTA · Weighted Ensemble

### Cell 4.1 — Transforms & Splits

In [ ]:
def _build_tta_transforms(img_size=IMG_SIZE):
    """One function — used in training, evaluation, and inference. No duplication."""
    ev = T.Compose([T.Resize((img_size,img_size)), T.ToTensor(), T.Normalize(MEAN,STD)])
    return [
        ev,
        T.Compose([T.Resize((img_size,img_size)), T.RandomHorizontalFlip(p=1.0),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(img_size*1.15),int(img_size*1.15))), T.CenterCrop(img_size),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(img_size*.85),int(img_size*.85))),
                   T.Pad(int(img_size*.075),fill=0), T.Resize((img_size,img_size)),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((img_size,img_size)), T.RandomRotation((10,10)),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((img_size,img_size)), T.RandomRotation((-10,-10)),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(img_size*1.08),int(img_size*1.08))), T.CenterCrop(img_size),
                   T.ToTensor(), T.Normalize(MEAN,STD)]),
    ]

train_tf = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.RandomHorizontalFlip(.5), T.RandomVerticalFlip(.2),
    T.RandomRotation(20),
    T.RandomAffine(0, translate=(.1,.1), scale=(.80,1.20), shear=10),
    T.ColorJitter(.4,.4,.3,.1),
    T.GaussianBlur(3, sigma=(.1,2.)),
    T.RandomGrayscale(p=0.15),
    T.ToTensor(), T.Normalize(MEAN,STD),
    T.RandomErasing(p=0.3, scale=(.02,.15)),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])
tta_tfs = _build_tta_transforms(IMG_SIZE)

trainval_df, test_df = train_test_split(df, test_size=.10, random_state=SEED, stratify=df['y'])
trainval_df = trainval_df.reset_index(drop=True)
test_df     = test_df.reset_index(drop=True)
fold_splits = list(StratifiedKFold(5, shuffle=True, random_state=SEED)
                   .split(trainval_df['path'], trainval_df['y']))
log.info(f'Train+Val: {len(trainval_df):,} | Test: {len(test_df):,} | TTA: {len(tta_tfs)} views')

### Cell 4.2 — Model, Dataset & Class Weights

In [ ]:
# ── FIX 2: single image read per __getitem__ ─────────────────────────────
class ImgDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        try:
            img = Image.open(self.df.loc[i,'path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE,IMG_SIZE), (128,128,128))
        return self.tf(img), int(self.df.loc[i,'y'])

class ImgDS_TTA(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        try:
            img = Image.open(self.df.loc[i,'path']).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE,IMG_SIZE), (128,128,128))
        return img, int(self.df.loc[i,'y'])

def collate_pil(batch):
    imgs,ys = zip(*batch); return list(imgs), torch.tensor(ys)

def balance_train_df(tr_df, seed=SEED):
    rng = np.random.default_rng(seed)
    cap = int(np.floor(2 * tr_df['y'].value_counts().mean()))
    return pd.concat([
        tr_df.loc[rng.choice(g.index.to_numpy(), size=cap, replace=(len(g)<cap))]
        for _,g in tr_df.groupby('y')
    ]).sample(frac=1., random_state=seed).reset_index(drop=True)

def build_model(num_classes, dropout=0.30):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_f = m.classifier[-1].in_features
    m.classifier[-1] = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha,alpha) if alpha>0 else 1.
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

def mixup_criterion(crit, pred, ya, yb, lam):
    return lam*crit(pred,ya) + (1-lam)*crit(pred,yb)

def compute_class_weights(df_train, num_classes, dev):
    counts = df_train['y'].value_counts().sort_index()
    total  = len(df_train)
    w = torch.tensor([total/(num_classes*counts.get(i,1)) for i in range(num_classes)],
                      dtype=torch.float32).to(dev)
    return w / w.mean()

class_weights = compute_class_weights(trainval_df, len(classes), device)
log.info(f'Class weights | min={class_weights.min():.3f} max={class_weights.max():.3f}')
_tmp = build_model(len(classes))
log.info(f'ConvNeXt-Tiny: {sum(p.numel() for p in _tmp.parameters()):,} params')
del _tmp

### Cell 4.3 — Eval Helpers

In [ ]:
# ── FIX 5: inference_mode everywhere in evaluation ────────────────────────
@torch.inference_mode()
def eval_f1_macro(model, loader):
    model.eval(); ys,preds=[],[]
    for x,y in loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
        ys.extend(y.tolist())
    _,_,f1,_=precision_recall_fscore_support(ys,preds,average='macro',zero_division=0)
    return float(f1)

@torch.inference_mode()
def eval_acc(model, loader):
    model.eval(); ys,preds=[],[]
    for x,y in loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
        ys.extend(y.tolist())
    return accuracy_score(ys,preds)

def print_metrics(y_true, y_pred, title):
    acc  = accuracy_score(y_true,y_pred)
    bacc = balanced_accuracy_score(y_true,y_pred)
    prec,rec,f1,_ = precision_recall_fscore_support(y_true,y_pred,average='macro',zero_division=0)
    log.info(f'{"═"*52}')
    log.info(f'  {title}')
    log.info(f'  Accuracy          : {acc:.4f}')
    log.info(f'  Balanced Accuracy : {bacc:.4f}')
    log.info(f'  Precision (macro) : {prec:.4f}')
    log.info(f'  Recall    (macro) : {rec:.4f}')
    log.info(f'  F1        (macro) : {f1:.4f}')
    log.info(f'{"═"*52}')
    return dict(acc=acc,bacc=bacc,prec=prec,rec=rec,f1=f1)

log.info('Eval helpers ready ✅')

### Cell 4.4 — CVV Training Loop

In [ ]:
def train_one_fold(fold_id, tr_idx, va_idx, epochs=25, patience=6, bs=64, lr=2e-4):
    set_seed(SEED+fold_id)
    tr_df  = trainval_df.iloc[tr_idx].reset_index(drop=True)
    va_df  = trainval_df.iloc[va_idx].reset_index(drop=True)
    tr_bal = balance_train_df(tr_df, seed=SEED+fold_id)

    tr_l = DataLoader(ImgDS(tr_bal,train_tf), bs, shuffle=True,
                       num_workers=2, pin_memory=PIN_MEMORY)
    va_l = DataLoader(ImgDS(va_df,eval_tf),   bs, shuffle=False,
                       num_workers=2, pin_memory=PIN_MEMORY)

    model = build_model(len(classes)).to(device)
    crit  = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.10)

    hp = list(model.classifier.parameters())
    bp = [p for n,p in model.named_parameters() if 'classifier' not in n]
    # head: higher weight_decay (new layer, prone to overfit)
    # backbone: lower weight_decay (pretrained, well-regularised)
    opt = optim.AdamW([{'params': hp, 'lr': lr,     'weight_decay': 0.10},
                        {'params': bp, 'lr': lr*0.1, 'weight_decay': 0.05}])
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler(device) if device=='cuda' else GradScaler('cpu')

    best_f1=best_acc=-1.; bad=0
    save_path = os.path.join(WORK_DIR, f'model_fold{fold_id}.pt')
    ll,ta_l,va_l2,vf_l = [],[],[],[]

    def set_backbone_grad(req):
        for n,p in model.named_parameters():
            if 'classifier' not in n: p.requires_grad_(req)
    set_backbone_grad(False)

    for ep in range(1, epochs+1):
        if ep==3: set_backbone_grad(True); log.info(f'[F{fold_id}] backbone unfrozen ep{ep}')
        model.train(); ep_l=[]; cp,ct=[],[]
        for x,y in tqdm(tr_l, desc=f'F{fold_id} Ep{ep:02d}', leave=False):
            x,y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            if random.random() < 0.5:
                xm,ya,yb,lam = mixup_data(x,y,.3)
                with autocast(device_type=device):
                    loss = mixup_criterion(crit,model(xm),ya,yb,lam)
            else:
                with autocast(device_type=device):
                    logits=model(x); loss=crit(logits,y)
                cp.extend(logits.argmax(1).cpu().tolist())
                ct.extend(y.cpu().tolist())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            ep_l.append(loss.item())
        sched.step()
        avg_l = float(np.mean(ep_l))
        ta    = accuracy_score(ct,cp) if ct else 0.
        va    = eval_acc(model,va_l)
        vf    = eval_f1_macro(model,va_l)
        log.info(f'[F{fold_id}] ep={ep:02d} loss={avg_l:.4f} '
                 f'tr={ta:.4f} val={va:.4f} f1={vf:.4f}')
        ll.append(avg_l); ta_l.append(ta); va_l2.append(va); vf_l.append(vf)
        if vf > best_f1+1e-4:
            best_f1,best_acc,bad = vf,va,0
            torch.save({'model':model.state_dict(),'classes':classes,
                         'fold':fold_id,'best_val_f1':best_f1}, save_path)
        else:
            bad+=1
            if bad>=patience:
                log.info(f'[F{fold_id}] Early stop @ ep{ep} best_f1={best_f1:.4f}'); break

    return save_path, best_f1, best_acc, ll, ta_l, va_l2, vf_l

log.info('train_one_fold ready ✅')

### Cell 4.5 — Run CVV Training

In [ ]:
model_paths=[]; fold_best_f1=[]; fold_best_acc=[]; fold_histories=[]

for fold_id,(tr_idx,va_idx) in enumerate(fold_splits):
    log.info(f'{"━"*50}  FOLD {fold_id}  {"━"*50}')
    p,bf1,bacc,ll,ta,va,vf = train_one_fold(fold_id,tr_idx,va_idx)
    model_paths.append(p); fold_best_f1.append(bf1); fold_best_acc.append(bacc)
    fold_histories.append({'loss':ll,'train_acc':ta,'val_acc':va,'val_f1':vf})

ensemble_weights = [f/sum(fold_best_f1) for f in fold_best_f1]
with open(TRAINING_STATE,'w') as f:
    json.dump({'model_paths':model_paths,'fold_best_f1':fold_best_f1,
               'fold_best_acc':fold_best_acc,'classes':list(classes),
               'class_to_idx':class_to_idx,'fold_histories':fold_histories}, f, indent=2)

log.info(f'CVV Training done ✅  Mean F1={np.mean(fold_best_f1):.4f} ± {np.std(fold_best_f1):.4f}')

### Cell 4.6 — Ensemble Evaluation

In [ ]:
# ── FIX 4: robust model loading with clear architecture mismatch error ────
@functools.lru_cache(maxsize=1)
def _load_models_cached(paths_tuple):
    ms = []
    for p in paths_tuple:
        ck = torch.load(p, map_location=device, weights_only=True)
        m  = build_model(len(classes)).to(device)
        try:
            m.load_state_dict(ck['model'])
        except RuntimeError as e:
            raise RuntimeError(
                f'Architecture mismatch loading {p}.\n'
                f'Saved classes={len(ck.get("classes",[]))} | '
                f'Current classes={len(classes)}.\n'
                f'Original error: {e}'
            )
        m.eval(); ms.append(m)
    return ms

def load_models(paths): return _load_models_cached(tuple(paths))

test_loader     = DataLoader(ImgDS(test_df,eval_tf), 64, shuffle=False,
                              num_workers=2, pin_memory=PIN_MEMORY)
test_loader_tta = DataLoader(ImgDS_TTA(test_df), 32, shuffle=False,
                              num_workers=2, pin_memory=PIN_MEMORY,
                              collate_fn=collate_pil)

# ── FIX 5: inference_mode in all predict functions ────────────────────────
@torch.inference_mode()
def weighted_tta_predict(paths, loader_tta, weights):
    ms=load_models(paths); yt,yp=[],[]
    for imgs,y in loader_tta:
        bp=None
        for tf in tta_tfs:
            x  = torch.stack([tf(im) for im in imgs]).to(device)
            wp = sum(w*torch.softmax(m(x),1) for m,w in zip(ms,weights))
            bp = wp if bp is None else bp+wp
        yp.extend((bp/len(tta_tfs)).argmax(1).cpu().tolist())
        yt.extend(y.tolist())
    return np.array(yt), np.array(yp)

@torch.inference_mode()
def soft_voting_predict(paths, loader):
    ms=load_models(paths); yt,yp=[],[]
    for x,y in loader:
        x=x.to(device)
        p=sum(torch.softmax(m(x),1) for m in ms)
        yp.extend((p/len(ms)).argmax(1).cpu().tolist())
        yt.extend(y.tolist())
    return np.array(yt), np.array(yp)

log.info('Running ensemble inference ...')
y_true_sv,  y_pred_sv  = soft_voting_predict(model_paths, test_loader)
y_true_wta, y_pred_wta = weighted_tta_predict(model_paths, test_loader_tta, ensemble_weights)
res_sv  = print_metrics(y_true_sv,  y_pred_sv,  'CVV-SV  | Soft Voting')
res_wta = print_metrics(y_true_wta, y_pred_wta, 'CVV-TTA | Weighted TTA (7 views)')

### Cell 4.7 — 📊 Step 4 Visualisation

In [ ]:
_,_,f1_per_class,support = precision_recall_fscore_support(
    y_true_wta, y_pred_wta, average=None, zero_division=0)

fig = plt.figure(figsize=(24,20))
fig.suptitle('Step 4 — CVV Results Dashboard', fontsize=16, fontweight='bold')

ax1=fig.add_subplot(3,3,1)
x=np.arange(2)
ax1.bar(x-.2,[res_sv['acc'],res_wta['acc']],.35,label='Accuracy',color='steelblue')
ax1.bar(x+.2,[res_sv['f1'], res_wta['f1']] ,.35,label='F1 macro', color='tomato')
for i,(a,f) in enumerate(zip([res_sv['acc'],res_wta['acc']],[res_sv['f1'],res_wta['f1']])):
    ax1.text(i-.2,a+.005,f'{a:.3f}',ha='center',fontsize=9)
    ax1.text(i+.2,f+.005,f'{f:.3f}',ha='center',fontsize=9)
ax1.set_xticks(x); ax1.set_xticklabels(['CVV-SV','CVV-TTA']); ax1.set_ylim(0,1)
ax1.set_title('Method Comparison'); ax1.legend()

ax2=fig.add_subplot(3,3,2)
xf=np.arange(len(fold_best_f1))
ax2.bar(xf-.2,fold_best_acc,.35,label='Val Acc',color='steelblue',alpha=.85)
ax2.bar(xf+.2,fold_best_f1, .35,label='Val F1', color='tomato',  alpha=.85)
ax2.axhline(np.mean(fold_best_f1),color='tomato',  ls='--',lw=1.5,label=f'MeanF1={np.mean(fold_best_f1):.3f}')
ax2.axhline(np.mean(fold_best_acc),color='steelblue',ls='--',lw=1.5,label=f'MeanAcc={np.mean(fold_best_acc):.3f}')
ax2.set_xticks(xf); ax2.set_xticklabels([f'F{i}' for i in range(len(fold_best_f1))])
ax2.set_ylim(0,1); ax2.set_title('CV Folds'); ax2.legend(fontsize=7)

ax3=fig.add_subplot(3,3,3)
for fi,h in enumerate(fold_histories):
    ax3.plot(h['val_f1'], label=f'Fold {fi}', alpha=.8)
ax3.set_title('Val F1 per Fold'); ax3.legend(fontsize=7); ax3.set_ylim(0,1); ax3.set_xlabel('Epoch')

ax4=fig.add_subplot(3,1,2)
ax4.bar(classes,f1_per_class,color=plt.cm.RdYlGn(f1_per_class),edgecolor='none')
ax4.axhline(f1_per_class.mean(),color='dodgerblue',ls='--',lw=1.5,label=f'Mean={f1_per_class.mean():.3f}')
ax4.axhline(0.80,color='orange',ls=':',lw=1,label='Target=0.80')
ax4.set_title(f'Per-Class F1 ({len(classes)} classes)'); ax4.set_ylim(0,1.05)
ax4.legend(fontsize=9); plt.sca(ax4); plt.xticks(rotation=90,fontsize=6)

ax5=fig.add_subplot(3,3,7)
sc=ax5.scatter(support,f1_per_class,c=f1_per_class,cmap='RdYlGn',alpha=.7,s=40,vmin=0,vmax=1)
plt.colorbar(sc,ax=ax5,label='F1')
for i,(s,f) in enumerate(zip(support,f1_per_class)):
    if f<0.60 or s<15: ax5.annotate(classes[i],(s,f),fontsize=6,alpha=.8)
ax5.set_xlabel('# Test Samples'); ax5.set_ylabel('F1'); ax5.set_title('F1 vs Support')

ax6=fig.add_subplot(3,3,8)
wrong = [(y_true_wta[i],y_pred_wta[i]) for i in range(len(y_true_wta)) if y_true_wta[i]!=y_pred_wta[i]]
top10 = Counter(wrong).most_common(10)
if top10:
    ax6.barh([f'{classes[t]}→{classes[p]}' for (t,p),_ in top10][::-1],
              [c for _,c in top10[::-1]], color='tomato',alpha=.8)
    ax6.set_title('Top-10 Confusions'); ax6.set_xlabel('# errors')

ax7=fig.add_subplot(3,3,9)
ax7.bar([f'F{i}' for i in range(len(ensemble_weights))],ensemble_weights,color='mediumseagreen')
ax7.set_title('Ensemble Weights'); ax7.set_ylabel('Weight')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,'step4_dashboard.png'),dpi=120,bbox_inches='tight')
plt.show()
log.info('Step 4 visualisation done')

### ⚡ ENDPOINT 4 — Resume from Step 4

In [ ]:
for _p,_n in [(STEP1_STATE,'Step1'),(TRAINING_STATE,'Training')]:
    if not os.path.exists(_p): raise FileNotFoundError(f'{_n} state missing: {_p}')

with open(STEP1_STATE)   as f: _s1 = json.load(f)
with open(TRAINING_STATE) as f: _s4 = json.load(f)

classes          = _s1['classes']
class_to_idx     = _s1['class_to_idx']
idx_to_class     = {int(i):c for c,i in class_to_idx.items()}
model_paths      = [os.path.join(WORK_DIR,os.path.basename(p)) for p in _s4['model_paths']]
fold_best_f1     = _s4['fold_best_f1']
fold_best_acc    = _s4['fold_best_acc']
fold_histories   = _s4.get('fold_histories',[])
ensemble_weights = [f/sum(fold_best_f1) for f in fold_best_f1]

missing = [p for p in model_paths if not os.path.exists(p)]
if missing: raise FileNotFoundError('Missing models:\n'+chr(10).join(missing))

eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])
tta_tfs = _build_tta_transforms(IMG_SIZE)

sam_finetuned = SAM_BEST_PATH if os.path.exists(SAM_BEST_PATH) else None
seg_cfg       = SegConfig(sam_checkpoint=sam_finetuned or SAM_CKPT)
sam_generator = load_sam_auto(seg_cfg)

rows = [(os.path.join(GLYPH2025_DIR,cls,fn),cls)
        for cls in classes for fn in os.listdir(os.path.join(GLYPH2025_DIR,cls))
        if fn.lower().endswith(IMG_EXTS)]
df = pd.DataFrame(rows,columns=['path','label']); df['y'] = df['label'].map(class_to_idx)

log.info('━'*55)
log.info('  ⚡  ENDPOINT 4 READY')
log.info('━'*55)
log.info(f'  Classes   : {len(classes)}')
log.info(f'  Models    : {len(model_paths)} folds ✅')
log.info(f'  Mean F1   : {np.mean(fold_best_f1):.4f}')
log.info(f'  SAM       : {"fine-tuned ✅" if sam_finetuned else "base ⚠️"}')

---
# 🚀 STEP 5 — End-to-End Inference
> Stela image → Gardiner codes → ready for Arabic NLP pipeline

### Cell 5.1 — predict_stela

In [ ]:
# ── FIX 5: inference_mode throughout ─────────────────────────────────────
@torch.inference_mode()
def predict_stela(image_path: str, show: bool = True,
                   conf_threshold: float = 0.30) -> list:
    """
    Full end-to-end pipeline: image → list of (crop, gardiner_code, confidence).

    Args:
        image_path     : Path to stela image
        show           : Show segmentation + classification plots
        conf_threshold : Drop predictions below this confidence

    Returns:
        list of (np.ndarray crop, str code, float confidence)
    """
    log.info(f'predict_stela: {os.path.basename(image_path)}')

    # Step 1: Segmentation
    try:
        ordered_masks, crops = run_single(
            image_path, cfg=seg_cfg, generator=sam_generator, show=show)
    except Exception as e:
        log.error(f'Segmentation failed: {e}'); return []
    if not crops:
        log.warning('No glyphs detected.'); return []

    # Step 2: Classification
    ms      = load_models(model_paths)   # cached — zero reload cost
    results = []; ignored = 0

    for i, crop in enumerate(crops):
        try:
            pil = Image.fromarray(crop.astype(np.uint8)).convert('RGB')
            bp  = None
            for tf in tta_tfs:
                x  = tf(pil).unsqueeze(0).to(device)
                wp = sum(w*torch.softmax(m(x),1) for m,w in zip(ms,ensemble_weights))
                bp = wp if bp is None else bp+wp
            probs    = (bp/len(tta_tfs)).squeeze()
            conf,pi  = probs.max(0)
            conf_val = conf.item()
            code     = idx_to_class[pi.item()]
        except Exception as e:
            log.warning(f'Glyph {i+1} failed: {e}'); ignored+=1; continue

        if conf_val < conf_threshold:
            log.info(f'  {i+1:>3} → IGNORED (conf={conf_val:.3f})')
            ignored += 1
        else:
            results.append((crop, code, conf_val))
            log.info(f'  {i+1:>3} → {code:>6}  conf={conf_val:.3f}')

    codes = [r[1] for r in results]
    log.info(f'{len(crops)} segmented | {len(results)} classified | {ignored} ignored')
    log.info(f'Gardiner codes: {codes}')

    if show and results:
        n=len(results); cols=min(n,10); rows_n=(n+cols-1)//cols
        fig,axes=plt.subplots(rows_n,cols,figsize=(cols*2.2,rows_n*2.5))
        axes=np.array(axes).flatten()
        for i,(crop,c,conf) in enumerate(results):
            axes[i].imshow(crop)
            axes[i].set_title(f'{i+1}. {c}\n{conf:.2f}',fontsize=8,
                               color='green' if conf>.7 else 'orange')
            axes[i].axis('off')
            for sp in axes[i].spines.values():
                sp.set_edgecolor(plt.cm.RdYlGn(conf)[:3]); sp.set_linewidth(2)
        for ax in axes[n:]: ax.axis('off')
        plt.suptitle(f'{len(results)}/{len(crops)} classified ({ignored} ignored)',
                      fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()

    return results

### Cell 5.2 — 📊 Step 5 Visualisation

In [ ]:
test_images = (glob.glob(os.path.join(SEG_DATA_DIR,'**','*.jpg'), recursive=True)
             + glob.glob(os.path.join(SEG_DATA_DIR,'**','*.png'), recursive=True))
test_images = [p for p in test_images if 'test' in p.lower() or 'stela' in p.lower()][:5]
if not test_images:
    test_images = (glob.glob(os.path.join(SEG_DATA_DIR,'**','*.jpg'), recursive=True)
                 + glob.glob(os.path.join(SEG_DATA_DIR,'**','*.png'), recursive=True))[:5]

all_results = []
for img_path in test_images:
    try:
        res = predict_stela(img_path, show=True, conf_threshold=0.30)
        all_results.extend(res)
    except Exception as e:
        log.warning(f'predict_stela failed {img_path}: {e}')

if all_results:
    confs      = [r[2] for r in all_results]
    codes_seen = Counter(r[1] for r in all_results)

    fig, axes = plt.subplots(1, 3, figsize=(20,5))
    fig.suptitle('Step 5 — Inference Summary', fontsize=14, fontweight='bold')

    axes[0].hist(confs, bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(0.30, color='tomato', ls='--', label='threshold=0.30')
    axes[0].axvline(np.mean(confs), color='green', ls='--', label=f'mean={np.mean(confs):.2f}')
    axes[0].set_title('Confidence Distribution'); axes[0].legend()

    top20 = codes_seen.most_common(20)
    axes[1].barh([c[0] for c in top20][::-1],[c[1] for c in top20][::-1],color='seagreen')
    axes[1].set_title('Top-20 Predicted Codes'); axes[1].set_xlabel('Count')

    high = sum(1 for c in confs if c>=.70); mid = sum(1 for c in confs if .30<=c<.70)
    axes[2].pie([high,mid],labels=[f'High≥0.70\n{high}',f'Mid 0.30-0.70\n{mid}'],
                 colors=['seagreen','steelblue'],autopct='%1.1f%%')
    axes[2].set_title('Confidence Tiers')

    plt.tight_layout()
    plt.savefig(os.path.join(WORK_DIR,'step5_inference_summary.png'),dpi=120)
    plt.show()
    log.info(f'{len(all_results)} glyphs classified across {len(test_images)} images')
else:
    log.info('No test images found — skip visualisation')

### ⚡ ENDPOINT 5 — Standalone Full Inference Session
> Paste this cell into a fresh Colab notebook after uploading model files.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  ENDPOINT 5 — Self-contained inference session
#  Prerequisites: all model_fold*.pt + step1_state.json + training_state.json
#                 uploaded to WORK_DIR
# ════════════════════════════════════════════════════════════════════════════
import os, json, logging, functools, random, glob
import numpy as np, torch, torch.nn as nn
import torchvision.transforms as T, torchvision.models as models
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('hiero')

WORK_DIR  = '/content/hiero'
IMG_SIZE  = 224
MEAN,STD  = [0.485,0.456,0.406],[0.229,0.224,0.225]
IMG_EXTS  = ('.jpg','.jpeg','.png','.bmp','.webp')
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
import psutil; PIN_MEMORY = torch.cuda.is_available() and (psutil.virtual_memory().total/(1024**3)>=8)

for _p,_n in [('step1_state.json','Step1'),('training_state.json','Training')]:
    fp = os.path.join(WORK_DIR,_p)
    if not os.path.exists(fp): raise FileNotFoundError(f'{_n} state missing: {fp}')

with open(os.path.join(WORK_DIR,'step1_state.json'))   as f: s1=json.load(f)
with open(os.path.join(WORK_DIR,'training_state.json')) as f: s4=json.load(f)

classes          = s1['classes']
class_to_idx     = s1['class_to_idx']
idx_to_class     = {int(i):c for c,i in class_to_idx.items()}
model_paths      = [os.path.join(WORK_DIR,os.path.basename(p)) for p in s4['model_paths']]
fold_best_f1     = s4['fold_best_f1']
ensemble_weights = [f/sum(fold_best_f1) for f in fold_best_f1]

missing=[p for p in model_paths if not os.path.exists(p)]
if missing: raise FileNotFoundError('Missing:\n'+chr(10).join(missing))

def _build_model(n):
    m=models.convnext_tiny(weights=None)
    m.classifier[-1]=nn.Sequential(nn.Dropout(.3),nn.Linear(m.classifier[-1].in_features,n))
    return m

def _build_tta(sz=IMG_SIZE):
    ev=T.Compose([T.Resize((sz,sz)),T.ToTensor(),T.Normalize(MEAN,STD)])
    return [ev,
        T.Compose([T.Resize((sz,sz)),T.RandomHorizontalFlip(1.),T.ToTensor(),T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(sz*1.15),int(sz*1.15))),T.CenterCrop(sz),T.ToTensor(),T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(sz*.85),int(sz*.85))),T.Pad(int(sz*.075),fill=0),T.Resize((sz,sz)),T.ToTensor(),T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((sz,sz)),T.RandomRotation((10,10)),T.ToTensor(),T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((sz,sz)),T.RandomRotation((-10,-10)),T.ToTensor(),T.Normalize(MEAN,STD)]),
        T.Compose([T.Resize((int(sz*1.08),int(sz*1.08))),T.CenterCrop(sz),T.ToTensor(),T.Normalize(MEAN,STD)]),
    ]
tta_tfs = _build_tta()

@functools.lru_cache(maxsize=1)
def _cached_models(paths_t):
    ms=[]
    for p in paths_t:
        ck=torch.load(p,map_location=device,weights_only=True)
        m=_build_model(len(classes)).to(device)
        try:
            m.load_state_dict(ck['model'])
        except RuntimeError as e:
            raise RuntimeError(
                f'Architecture mismatch in {p}. '
                f'Saved n_classes={len(ck.get("classes",[]))} vs current={len(classes)}.\n{e}')
        m.eval(); ms.append(m)
    return ms

# SAM & segmentation (re-run Cell 3.1-3.3 if needed, or import if available)
SAM_BEST = os.path.join(WORK_DIR,'sam_finetuned','best_sam_hiero.pth')
SAM_CKPT = os.path.join(WORK_DIR,'sam_vit_b.pth')
# seg_cfg and sam_generator must be available from Cell 3
# If in a fresh session, run Cell 3.1-3.3 before calling predict_stela

@torch.inference_mode()
def predict_stela_ep5(image_path, show=True, conf_threshold=0.30):
    from PIL import Image as _PIL
    ms = _cached_models(tuple(model_paths))
    try: ordered_masks, crops = run_single(image_path, cfg=seg_cfg, generator=sam_generator, show=show)
    except Exception as e: log.error(f'Seg failed: {e}'); return []
    if not crops: return []
    results=[]; ignored=0
    for i,crop in enumerate(crops):
        try:
            pil=_PIL.fromarray(crop.astype(np.uint8)).convert('RGB')
            bp=None
            for tf in tta_tfs:
                x=tf(pil).unsqueeze(0).to(device)
                wp=sum(w*torch.softmax(m(x),1) for m,w in zip(ms,ensemble_weights))
                bp=wp if bp is None else bp+wp
            probs=(bp/len(tta_tfs)).squeeze(); conf,pi=probs.max(0)
            conf_val=conf.item(); code=idx_to_class[pi.item()]
        except Exception as e: log.warning(f'Glyph {i+1}: {e}'); ignored+=1; continue
        if conf_val<conf_threshold: ignored+=1
        else: results.append((crop,code,conf_val))
    log.info(f'{len(crops)} seg | {len(results)} classified | {ignored} ignored')
    log.info(f'Codes: {[r[1] for r in results]}')
    return results

log.info('━'*55)
log.info('  ⚡  ENDPOINT 5 — INFERENCE READY')
log.info('━'*55)
log.info(f'  Classes  : {len(classes)}')
log.info(f'  Models   : {len(model_paths)} folds  Mean F1={np.mean(fold_best_f1):.4f}')
log.info(f'  SAM      : {"fine-tuned ✅" if os.path.exists(SAM_BEST) else "base ⚠️"}')
log.info('')
log.info('  Usage:')
log.info('    results = predict_stela_ep5("/path/to/stela.jpg")')
log.info('    codes   = [r[1] for r in results]  # feed to NLP')
log.info('━'*55)